# NEWS CATEGORY CLASSIFICATION

#### DATASET DOWNLOADING

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rmisra/news-category-dataset")

print("Path to dataset files:", path)

p:\PROJECTS\DATA SCIENCE\NEWS CATEGORY CLASSIFICATION\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\khage\.cache\kagglehub\datasets\rmisra\news-category-dataset\versions\3


#### DATASET LOADING

In [2]:
import pandas as pd
import json

path+="/News_Category_Dataset_v3.json"
df = pd.read_json(path, lines=True)

#### DATA INSPECTION

GETTING SHAPE OF THE DATASET

In [3]:
df.shape

(209527, 6)

DISPLAYING FIRST 5 RECORDS


In [4]:
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


GETTING HOW MANY RECORDS ARE THERE IN EACH CATEGORY

In [5]:
df["category"] = df["category"].replace({
    "THE WORLDPOST": "WORLDPOST",
    "STYLE": "STYLE & BEAUTY",
    "ARTS": "ARTS & CULTURE",
    "CULTURE & ARTS": "ARTS & CULTURE",
    "PARENTS": "PARENTING"
})

In [6]:
print(df['category'].value_counts())

category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
PARENTING         12746
STYLE & BEAUTY    12068
TRAVEL             9900
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
WORLDPOST          6243
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
ARTS & CULTURE     3922
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
RELIGION           2577
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
COLLEGE            1144
LATINO VOICES      1130
EDUCATION          1014
Name: count, dtype: int64


GETTING TOTAL RECORDS AND CATEGORY COUNTS

In [7]:
print(f"Total records: {len(df)}")
print(f"Categories: {df['category'].nunique()}")

Total records: 209527
Categories: 37


#### DATA CLEANING

CHECKING NULL OCCURANCES

In [8]:
df.isnull().sum()

link                 0
headline             0
category             0
short_description    0
authors              0
date                 0
dtype: int64

CHECKING DUPLICATE VALUES

In [9]:
print(df.duplicated().sum())
dupRow = df[df.duplicated()]
print(dupRow)

13
                                                     link  \
67677   https://www.huffingtonpost.comhttp://www.mothe...   
67923   https://www.huffingtonpost.comhttp://gizmodo.c...   
70239   https://www.huffingtonpost.comhttp://www.cnbc....   
139830  https://www.huffingtonpost.comhttp://www.cnn.c...   
144409  https://www.huffingtonpost.comhttp://www.upwor...   
145142  https://www.huffingtonpost.comhttp://www.weath...   
178155  https://www.huffingtonpost.comhttp://www.busin...   
187329  https://www.huffingtonpost.comhttp://www.nytim...   
194596  https://www.huffingtonpost.comhttp://blogs.wsj...   
194598  https://www.huffingtonpost.comhttp://www.theda...   
207122  https://www.huffingtonpost.comhttp://d.repubbl...   
207208  https://www.huffingtonpost.comhttp://d.repubbl...   
207318  https://www.huffingtonpost.comhttp://d.repubbl...   

                                                 headline        category  \
67677   On Facebook, Trump's Longtime Butler Calls For...        

REMOVING DUPLICATE VALUES

In [10]:
df = df.drop_duplicates()

CREATING NEW COLUMN AS "text" FOR BOTH HEADLINE AND DESCRIPTION

In [11]:
df['text'] = df['headline'] + ' ' + df['short_description']

REMOVING EMPTY TEXT

In [12]:
df = df[df['text'].str.strip() != ""]

REMOVING COLUMS THAT ARE NOT REQUIRED

In [13]:
df.drop(columns=['link','short_description','headline','authors'], inplace=True)

In [14]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["label"] = le.fit_transform(df["category"])

num_labels = len(le.classes_)


In [15]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

#### TOKENIZATION

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from transformers import DataCollatorWithPadding
from transformers import TrainerCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Trainer


In [17]:
train_dataset = Dataset.from_pandas(
    train_df
)

test_dataset = Dataset.from_pandas(
    test_df
)

In [21]:
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer


MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# =====================================================
# TOKENIZATION
# =====================================================

def tokenize(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

# =====================================================
# DYNAMIC PADDING
# =====================================================

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

# =====================================================
# CLASS WEIGHTS
# =====================================================

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
)

print("\nClass Weights:")
print(class_weights)

# =====================================================
# LOAD MODEL
# =====================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels
)

# =====================================================
# METRICS
# =====================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

# =====================================================
# CUSTOM TRAINER
# =====================================================

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        **kwargs
    ):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        loss_function = nn.CrossEntropyLoss(
            weight=class_weights.to(
                model.device
            )
        )

        loss = loss_function(
            logits.view(
                -1,
                model.config.num_labels
            ),
            labels.view(-1)
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

# =====================================================
# TRAINING CONFIG
# =====================================================

training_args = TrainingArguments(

    output_dir="./news_classifier",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    warmup_ratio=0.1,

    lr_scheduler_type="cosine",

    weight_decay=0.01,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=6,

    fp16=True,

    load_best_model_at_end=True,

    metric_for_best_model="f1_macro",

    greater_is_better=True,

    logging_steps=100,

    report_to="none"
)

# =====================================================
# TRAINER
# =====================================================

trainer = WeightedTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

# =====================================================
# TRAIN
# =====================================================

trainer.train()

# =====================================================
# FINAL EVALUATION
# =====================================================

results = trainer.evaluate()

print("\nFinal Results")
print(results)


Map: 100%|██████████| 41902/41902 [00:01<00:00, 27495.26 examples/s]



Class Weights:
tensor([1.4436, 1.2357, 0.9451, 4.9507, 1.0486, 1.5894, 1.6527, 5.5856, 0.3262,
        3.9254, 4.0410, 0.8931, 4.0518, 2.1592, 0.8459, 1.3107, 1.6254, 5.0110,
        1.9243, 3.2241, 0.4442, 0.1591, 0.8922, 2.1969, 2.5665, 1.1152, 0.4693,
        2.7012, 2.6964, 0.5720, 4.1106, 1.5503, 2.0387, 0.3156, 1.5856, 1.7165,
        0.9073])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3077.60it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 